# Weather Prophet – From Raw Data to Predictions: A Full Walkthrough

In [6]:
import os

import numpy as np
import pandas as pd

import torch

print(torch.__version__)

2.7.0+cpu


## Shape Raw Data into Desired Format and Save as .pt File

In [9]:
csv_files: list[str] = os.listdir("./raw_data")

df_data: pd.DataFrame = pd.DataFrame()
df_labels: pd.DataFrame = pd.DataFrame()

for file_name in csv_files:
    current_df = pd.read_csv(os.path.join("./raw_data", file_name))
    df_data = pd.concat([df_data, current_df], ignore_index=True)

df_labels = df_data[["tempmax", "tempmin"]]

df_data = df_data.drop(
    columns=[
        "cloudcover",
        "conditions",
        "datetime",
        "description",
        "dew",
        "feelslikemax",
        "feelslikemin",
        "icon",
        "moonphase",
        "name",
        "precip",
        "precipcover",
        "precipprob",
        "preciptype",
        "severerisk",
        "snow",
        "snowdepth",
        "solarradiation",
        "solarenergy",
        "stations",
        "sunrise",
        "sunset",
        "tempmax",
        "tempmin",
        "uvindex",
        "visibility",
        "winddir",
        "windgust"
    ]
)

print(f"DATA\n {df_data.head()}\n")
print(f"LABELS\n {df_labels.head()}\n")

DATA
    temp  feelslike  humidity  windspeed  sealevelpressure
0  88.7       93.1      56.5       15.7            1004.1
1  86.8       92.2      58.9       14.8            1004.3
2  87.1       93.5      60.1       16.3            1005.0
3  86.8       92.1      58.4       14.3            1005.5
4  86.4       93.0      63.1       18.1            1004.5

LABELS
    tempmax  tempmin
0    101.0     80.3
1     91.7     84.5
2     90.2     84.8
3     88.1     85.7
4     88.2     84.1



In [10]:

tempmax = df_labels["tempmax"].to_numpy()
tempmin = df_labels["tempmin"].to_numpy()

new_labels = []
num_days = len(df_labels)
forecast_horizon = 5

for i in range(num_days - forecast_horizon):
    future_temps = []
    for day_ahead in range(1, forecast_horizon + 1):
        future_temps.append(tempmax[i + day_ahead])
        future_temps.append(tempmin[i + day_ahead])
    new_labels.append(future_temps)

new_labels = np.array(new_labels, dtype=np.float32)
new_data = df_data.iloc[:num_days - forecast_horizon].to_numpy(dtype=np.float32)

torch.save(
    {
        "data": new_data,
        "labels": new_labels
    }, 
    "./pytorch_data/dataset.pt"
)

## Define a Custom PyTorch Dataset Class

In [11]:
from torch.utils.data import Dataset

class WeatherDataset(Dataset):
    def __init__(self, X_data, Y_data):
        """
        Args:
            X_data: numpy array or tensor of shape (N_samples, 5) with daily weather features
            Y_data: numpy array or tensor of shape (N_samples, 10) with min/max temps for next 5 days
        """

        if not torch.is_tensor(X_data):
            X_data = torch.tensor(X_data, dtype=torch.float32)
        if not torch.is_tensor(Y_data):
            Y_data = torch.tensor(Y_data, dtype=torch.float32)
            
        self.X_data = X_data
        self.Y_data = Y_data
    
    def __len__(self):
        return len(self.X_data)
    
    def __getitem__(self, idx):
        return self.X_data[idx], self.Y_data[idx]


## Define the PyTorch Model for Weather Prediction

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class WeatherProphet(nn.Module):
    def __init__(self):
        super(WeatherProphet, self).__init__()
        
        # Input size: 5 features
        # Output size: 10 (5 days * 2 values per day)
        
        self.fc1 = nn.Linear(5, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        output = self.fc3(x)
        return output

## Load Data, Initialize Model, Loss Function, and Optimizer

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

data = torch.load("./pytorch_data/dataset.pt", weights_only=False)
X_data = torch.tensor(data['data'])
Y_data = torch.tensor(data['labels'])

dataset = TensorDataset(X_data, Y_data)

batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [15]:
import torch.optim as optim

model = WeatherProphet()

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training Loop

In [16]:
epochs = 100
for epoch in range(epochs):
    model.train()
    total_train_loss = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs, labels
        labels = labels.float()

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * inputs.size(0)

    avg_train_loss = total_train_loss / len(dataloader.dataset)

    model.eval()
    total_val_loss = 0

    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs, labels = inputs, labels
            labels = labels.float()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_val_loss += loss.item() * inputs.size(0)

    avg_val_loss = total_val_loss / len(dataloader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


Epoch 1/100 | Train Loss: 5763.9183 | Val Loss: 1576.4869
Epoch 2/100 | Train Loss: 1005.5178 | Val Loss: 543.5167
Epoch 3/100 | Train Loss: 367.2840 | Val Loss: 213.3270
Epoch 4/100 | Train Loss: 209.9464 | Val Loss: 200.4684
Epoch 5/100 | Train Loss: 180.8275 | Val Loss: 157.2157
Epoch 6/100 | Train Loss: 148.6389 | Val Loss: 140.2018
Epoch 7/100 | Train Loss: 137.9839 | Val Loss: 135.8918
Epoch 8/100 | Train Loss: 135.7826 | Val Loss: 135.0186
Epoch 9/100 | Train Loss: 134.4140 | Val Loss: 132.5698
Epoch 10/100 | Train Loss: 133.2665 | Val Loss: 131.9782
Epoch 11/100 | Train Loss: 132.8472 | Val Loss: 131.6741
Epoch 12/100 | Train Loss: 132.5224 | Val Loss: 131.5624
Epoch 13/100 | Train Loss: 132.0328 | Val Loss: 130.9723
Epoch 14/100 | Train Loss: 131.8642 | Val Loss: 130.7725
Epoch 15/100 | Train Loss: 130.7319 | Val Loss: 130.6019
Epoch 16/100 | Train Loss: 130.1136 | Val Loss: 130.2839
Epoch 17/100 | Train Loss: 129.2530 | Val Loss: 129.0916
Epoch 18/100 | Train Loss: 129.0858 |

## Save Model State Dictionary

In [17]:
torch.save(model.state_dict(), "./pytorch_data/weather_prophet_model.pth")

## Test Model Prediction Accuracy

In [18]:
def print_pred(model, vals, ans):
    with torch.inference_mode():
        vals = vals.clone().detach().float()
        vals = vals.view(1, -1)

        pred = model(vals)
        pred_np = pred.cpu().numpy()
        pred_rounded = np.round(pred_np, 1)
        
        diff = np.round(np.abs(pred_rounded.flatten() - ans.flatten()), 1)

        print(f"\nPredicted:   {pred_rounded.flatten()}")
        print(f"Actual:      {ans.flatten()}")
        print(f"Difference:  {diff}\n")

In [20]:
dataset = torch.load("./pytorch_data/dataset.pt", weights_only=False)

test_model = WeatherProphet()
test_model.load_state_dict(torch.load("./pytorch_data/weather_prophet_model.pth", weights_only=False))
test_model.eval()

for i in range(5):
    sample = torch.tensor(dataset['data'][i])
    label = dataset['labels'][i]

    print(f"Day - {i + 1}")
    print_pred(test_model, sample, label)

Day - 1

Predicted:   [91.9 80.7 89.1 77.2 88.9 73.2 86.8 75.1 81.8 72.1]
Actual:      [91.7 84.5 90.2 84.8 88.1 85.7 88.2 84.1 85.  83.4]
Difference:  [ 0.2  3.8  1.1  7.6  0.8 12.5  1.4  9.   3.2 11.3]

Day - 2

Predicted:   [90.6 79.6 88.  76.3 88.  72.4 86.1 74.3 81.4 71.6]
Actual:      [90.2 84.8 88.1 85.7 88.2 84.1 85.  83.4 84.3 83.3]
Difference:  [ 0.4  5.2  0.1  9.4  0.2 11.7  1.1  9.1  2.9 11.7]

Day - 3

Predicted:   [91.1 80.1 88.4 76.8 88.3 72.8 86.6 74.7 81.5 72. ]
Actual:      [88.1 85.7 88.2 84.1 85.  83.4 84.3 83.3 86.4 83.2]
Difference:  [ 3.   5.6  0.2  7.3  3.3 10.6  2.3  8.6  4.9 11.2]

Day - 4

Predicted:   [90.6 79.7 88.1 76.3 88.  72.4 86.1 74.3 81.5 71.6]
Actual:      [88.2 84.1 85.  83.4 84.3 83.3 86.4 83.2 88.1 84.6]
Difference:  [ 2.4  4.4  3.1  7.1  3.7 10.9  0.3  8.9  6.6 13. ]

Day - 5

Predicted:   [90.5 79.5 87.8 76.4 87.8 72.5 86.4 74.4 81.2 71.8]
Actual:      [85.  83.4 84.3 83.3 86.4 83.2 88.1 84.6 87.5 85.7]
Difference:  [ 5.5  3.9  3.5  6.9  1.4 10

## Conclusion